In [15]:
# Step 1: Set Up Your Environment
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

In [16]:
# Initialize Spark session
spark = SparkSession.builder.appName("dogfood").getOrCreate()

In [17]:
# Step 2: Load the Data
data = spark.read.csv("dog_food.csv", header=True, inferSchema=True)
data.show(3)

+---+---+----+---+-------+
|  A|  B|   C|  D|Spoiled|
+---+---+----+---+-------+
|  4|  2|12.0|  3|    1.0|
|  5|  6|12.0|  7|    1.0|
|  6|  2|13.0|  6|    1.0|
+---+---+----+---+-------+
only showing top 3 rows



In [18]:
# Step 3: Data Preprocessing 
assembler = VectorAssembler(inputCols=["A", "B", "C", "D"], outputCol="features")

In [19]:
# Step 4: Apply Feature Scaling
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)

In [20]:
# Create pipeline for preprocessing
pipeline = Pipeline(stages=[assembler, scaler])
data_preprocessed = pipeline.fit(data).transform(data)

In [21]:
# Split the data into training and testing sets
train_data, test_data = data_preprocessed.randomSplit([0.8, 0.2], seed=1)

In [22]:
# Step 5: Build and Train the Classification Model
lr = LogisticRegression(featuresCol="scaled_features", labelCol="Spoiled")  # "Spoiled" is already numeric
lr_model = lr.fit(train_data)

In [23]:
# Step 6: Make Predictions
predictions = lr_model.transform(test_data)

In [24]:
# Step 7: Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="Spoiled", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy = {accuracy}")

Test Accuracy = 0.9891304347826086


In [25]:
# Step 8: Compute Confusion Matrix
confusion_matrix = predictions.groupBy("Spoiled", "prediction").count().orderBy("Spoiled", "prediction")
confusion_matrix.show()

+-------+----------+-----+
|Spoiled|prediction|count|
+-------+----------+-----+
|    0.0|       0.0|   69|
|    1.0|       0.0|    1|
|    1.0|       1.0|   22|
+-------+----------+-----+



In [26]:
# Extract TP, TN, FP, FN
tp = predictions.filter((col("Spoiled") == 1.0) & (col("prediction") == 1.0)).count()
tn = predictions.filter((col("Spoiled") == 0.0) & (col("prediction") == 0.0)).count()
fp = predictions.filter((col("Spoiled") == 0.0) & (col("prediction") == 1.0)).count()
fn = predictions.filter((col("Spoiled") == 1.0) & (col("prediction") == 0.0)).count()

# Print Confusion Matrix
print(f"Confusion Matrix:\n TP: {tp}, FP: {fp}\n FN: {fn}, TN: {tn}")

Confusion Matrix:
 TP: 22, FP: 0
 FN: 1, TN: 69


In [27]:
# Step 9: Save the Model
lr_model.save("/work/kripa/")

Py4JJavaError: An error occurred while calling o625.save.
: org.apache.spark.SparkException: Job aborted.
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:106)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopDataset$1(PairRDDFunctions.scala:1091)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:407)
	at org.apache.spark.rdd.PairRDDFunctions.saveAsHadoopDataset(PairRDDFunctions.scala:1089)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopFile$4(PairRDDFunctions.scala:1062)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOpera